# F2-M02: Calidad de Datos

**TFM: Predicción de Abandono Universitario**

| | |
|---|---|
| **Autora** | María José Morte |
| **Email** | mjmorteruiz@uoc.edu (UOC) \| morte@uji.es (UJI) |

---

## Qué hace
Análisis de calidad: duplicados exactos y por clave, estructura longitudinal
(filas por alumno), distribución por titulación/rama/sexo, barras apiladas
curso×sexo y detección de anomalías en texto.

## Requisitos
- `df_alumno.parquet` en `data/02_processed/`
- Módulos: `src.config`, `src.utils`, `src.html`

## Genera
- `docs/html/fase2/m02_calidad.html`
- `docs/html/fase2/graficos/m02_cursos.html`
- `docs/html/fase2/graficos/m02_filas_alumno.html`
- `docs/html/fase2/graficos/m02_titulaciones.html`
- `docs/html/fase2/graficos/m02_rama.html`
- `docs/html/fase2/graficos/m02_sexo.html`
- `docs/html/fase2/graficos/m02_curso_sexo.html` *(NUEVO)*
- `docs/html/fase2/graficos/m02_anomalias_barras.html`
- `docs/html/fase2/graficos/m02_anomalias_tabla.html`

## Flujo
```
M00 Índice → M01 Inspección → **M02 Calidad** → M03 Nulos → ...
```

## Siguiente
`f2_m03_nulos.ipynb`

In [1]:
# ============================================================================
# CELDA 1: CONFIGURACIÓN DEL ENTORNO
# ============================================================================
# - Detecta entorno (Colab / local)
# - Localiza ROOT buscando src/ (robusto, sin hardcodes)
# - Importa módulos del proyecto y librerías de visualización
# - Crea directorios de salida
# ============================================================================

import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

# --- Detectar entorno y localizar ROOT ---
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/AU_UJI')
else:
    _cwd = Path.cwd()
    ROOT = _cwd
    for _ in range(10):
        if (ROOT / 'src').is_dir():
            break
        ROOT = ROOT.parent
    else:
        raise FileNotFoundError(
            f"No se encontró carpeta 'src/' desde {_cwd}. "
            f"Verifica que el notebook está dentro de AU_UJI/"
        )

sys.path.insert(0, str(ROOT))

# --- Imports ---
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px

from src.config import RUTA_PROCESSED, RUTA_HTML, info_entorno
from src.utils import crear_directorios, formato_numero_es
from src.html import (
    generar_kpis_html,
    generar_seccion_html,
    generar_html_navegacion_completa,
    guardar_html
)
from src.html.render import render_base_html

# --- Rutas de salida ---
RUTA_FASE2 = RUTA_HTML / 'fase2'
RUTA_GRAFICOS = RUTA_FASE2 / 'graficos'
crear_directorios([RUTA_FASE2, RUTA_GRAFICOS])

info_entorno()

✓ Directorios verificados: 2
✓ ===========================================================================
✓ 📌 INFORMACIÓN DEL ENTORNO DEL PROYECTO
✓ ===========================================================================
✓ 🖥️  Entorno detectado: Local
✓ 📂 Ruta base:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI
✓ 📁 RAW:           C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw
✓ 📁 INTERIM:       C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\01_interim
✓ 📁 PROCESSED:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\02_processed
✓ 📁 FEATURES:      C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features
✓ 📁 AUTOML:        C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl
✓ 📁 NOTEBOOKS:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\notebooks
✓ 📄 Excel principal: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw\datos_proyecto_sin_preinscrip.xlsx
✓

In [2]:
# ============================================================================
# CELDA 2: CARGAR DATOS Y ANÁLISIS DE DUPLICADOS
# ============================================================================
# Carga el dataset y comprueba:
# - Duplicados exactos (todas las columnas)
# - Duplicados por clave (per_id_ficticio + curso)
# - Ratio filas/alumno (estructura longitudinal)
# ============================================================================

print('=' * 60)
print('ANÁLISIS DE CALIDAD')
print('=' * 60)

df = pd.read_parquet(RUTA_PROCESSED / 'df_alumno.parquet')

n_filas = len(df)

# --- Duplicados exactos ---
n_dup_exactos = df.duplicated().sum()

# --- Duplicados por clave (alumno + curso) ---
clave_cols = []
if 'per_id_ficticio' in df.columns:
    clave_cols.append('per_id_ficticio')
for col in ['curso_aca', 'curso_aca_id']:
    if col in df.columns:
        clave_cols.append(col)
        break

n_dup_clave = df.duplicated(subset=clave_cols).sum() if len(clave_cols) >= 2 else 0

# --- Alumnos y ratio ---
n_alumnos = df['per_id_ficticio'].nunique() if 'per_id_ficticio' in df.columns else n_filas
filas_por_alumno = n_filas / n_alumnos if n_alumnos > 0 else 1

print(f'✅ Registros: {formato_numero_es(n_filas)}')
print(f'👥 Alumnos: {formato_numero_es(n_alumnos)}')
print(f'📊 Filas/alumno: {filas_por_alumno:.1f}')

ANÁLISIS DE CALIDAD
✅ Registros: 109.568
👥 Alumnos: 30.872
📊 Filas/alumno: 3.5


In [3]:
# ============================================================================
# CELDA 3: GRÁFICOS — MATRÍCULAS POR CURSO + CURSOS POR ALUMNO
# ============================================================================
# Gráfico 1: Barras con matrículas por curso académico + línea de media.
# Gráfico 2: Distribución de cuántos cursos tiene cada alumno.
# ============================================================================

print('=' * 60)
print('GENERANDO GRÁFICOS')
print('=' * 60)

# --- Gráfico 1: Matrículas por curso ---
curso_col = None
for col in ['curso_aca', 'curso_aca_id']:
    if col in df.columns:
        curso_col = col
        break

if curso_col:
    cursos = df[curso_col].value_counts().sort_index()
    media_cursos = cursos.mean()

    fig_cursos = go.Figure(go.Bar(
        x=cursos.index.astype(str),
        y=cursos.values,
        marker_color='#3182ce',
        text=cursos.values,
        textposition='outside'
    ))
    fig_cursos.add_hline(
        y=media_cursos, line_dash='dash', line_color='#e53e3e',
        annotation_text=f'Media: {media_cursos:,.0f}', annotation_position='right'
    )
    fig_cursos.update_layout(
        title='📅 Registros por Curso Académico',
        xaxis_title='Curso Académico',
        yaxis_title='Número de Registros',
        height=380,
        margin=dict(t=60, b=50, l=60, r=40)
    )
    fig_cursos.write_html(RUTA_GRAFICOS / 'm02_cursos.html', include_plotlyjs='cdn')
    print('✅ Gráfico matrículas por curso')

# --- Gráfico 2: Cursos por alumno ---
if 'per_id_ficticio' in df.columns:
    filas_alumno = df.groupby('per_id_ficticio').size()
    dist_filas = filas_alumno.value_counts().sort_index()

    fig_filas = go.Figure(go.Bar(
        x=dist_filas.index.astype(str),
        y=dist_filas.values,
        marker_color='#38a169',
        text=dist_filas.values,
        textposition='outside'
    ))
    fig_filas.update_layout(
        title='📊 Cursos por Alumno',
        xaxis_title='Nº de Cursos',
        yaxis_title='Nº de Alumnos',
        height=350,
        margin=dict(t=60, b=50, l=60, r=40)
    )
    fig_filas.write_html(RUTA_GRAFICOS / 'm02_filas_alumno.html', include_plotlyjs='cdn')
    print('✅ Gráfico cursos por alumno')

GENERANDO GRÁFICOS
✅ Gráfico matrículas por curso
✅ Gráfico cursos por alumno


In [4]:
# ============================================================================
# CELDA 4: GRÁFICOS — TITULACIONES Y RAMA
# ============================================================================
# Gráfico 3: Top 10 titulaciones con nombres abreviados.
# Gráfico 4: Distribución por rama de conocimiento (donut).
# ============================================================================

def formatear_titulo(texto, max_chars=35):
    """Abrevia nombres largos de titulaciones para gráficos."""
    texto = str(texto)
    abreviaturas = {
        'Grado en ': 'G. ', 'Máster en ': 'M. ',
        'Ingeniería': 'Ing.', 'Administración': 'Admin.',
        'Desarrollo': 'Des.', 'Publicidad': 'Public.',
        'Comunicación': 'Comunic.', 'Traducción': 'Traduc.',
        'Interpretación': 'Interp.', 'Educación': 'Ed.',
        'Diseño': 'D.', 'Industrial': 'Ind.',
        'Productos': 'Prod.', 'Relaciones': 'Rel.',
        'Públicas': 'Públ.', 'Infantil': 'Inf.',
        'Primaria': 'Prim.', 'Contabilidad': 'Contab.',
        'Finanzas': 'Finan.', 'Audiovisual': 'Audiov.',
        'Psicología': 'Psicol.', 'Derecho': 'Der.',
        'Empresas': 'Empr.'
    }
    for largo, corto in abreviaturas.items():
        texto = texto.replace(largo, corto)

    if len(texto) <= max_chars:
        return texto
    medio = len(texto) // 2
    espacio = texto.rfind(' ', 0, medio + 8)
    if espacio != -1 and espacio > 5:
        return texto[:espacio] + '<br>' + texto[espacio+1:]
    return texto[:max_chars-2] + '..'

# --- Gráfico 3: Top titulaciones ---
if 'titulacion' in df.columns:
    tit_counts = df['titulacion'].value_counts().head(10).sort_values(ascending=True)
    labels_tit = [formatear_titulo(t) for t in tit_counts.index]

    fig_tit = go.Figure(go.Bar(
        x=tit_counts.values,
        y=labels_tit,
        orientation='h',
        marker_color='#805ad5',
        text=[f'{v:,}' for v in tit_counts.values],
        textposition='outside'
    ))
    fig_tit.update_layout(
        title='🎓 Top 10 Titulaciones',
        xaxis_title='Matrículas',
        height=500,
        margin=dict(t=60, b=50, l=220, r=60),
        bargap=0.15
    )
    fig_tit.update_xaxes(range=[0, tit_counts.max() * 1.18])
    fig_tit.write_html(RUTA_GRAFICOS / 'm02_titulaciones.html', include_plotlyjs='cdn')
    print('✅ Gráfico titulaciones')

# --- Gráfico 4: Rama de conocimiento ---
if 'rama' in df.columns:
    rama_counts = df['rama'].value_counts()
    colores_rama = ['#3182ce', '#38a169', '#ed8936', '#e53e3e', '#805ad5']

    fig_rama = go.Figure(go.Pie(
        labels=rama_counts.index,
        values=rama_counts.values,
        hole=0.4,
        marker_colors=colores_rama[:len(rama_counts)],
        textinfo='label+percent',
        textposition='outside'
    ))
    fig_rama.update_layout(
        title='📚 Distribución por Rama',
        height=400,
        margin=dict(t=60, b=40, l=40, r=40)
    )
    fig_rama.write_html(RUTA_GRAFICOS / 'm02_rama.html', include_plotlyjs='cdn')
    print('✅ Gráfico rama')

✅ Gráfico titulaciones
✅ Gráfico rama


In [5]:
# ============================================================================
# CELDA 5: GRÁFICOS — SEXO + BARRAS APILADAS CURSO×SEXO
# ============================================================================
# Gráfico 5: Donut distribución por sexo.
# Gráfico 6 (NUEVO): Barras apiladas curso×sexo para detectar
#   si la proporción hombre/mujer cambia por curso académico.
#   Idea del profesor: stacked bar para ver proporción temporal.
# ============================================================================

# --- Gráfico 5: Distribución por sexo ---
if 'sexo' in df.columns:
    sexo_counts = df['sexo'].value_counts().sort_index()

    labels_sexo = []
    for s in sexo_counts.index:
        if s == 1:
            labels_sexo.append('Hombre')
        elif s == 2:
            labels_sexo.append('Mujer')
        else:
            labels_sexo.append(str(s))

    fig_sexo = go.Figure(go.Pie(
        labels=labels_sexo,
        values=sexo_counts.values,
        hole=0.4,
        marker_colors=['#9567FE', '#FD6320'],  # Violeta=Hombre, Naranja=Mujer
        textinfo='label+percent',
        textposition='outside'
    ))
    fig_sexo.update_layout(
        title='👥 Distribución por Sexo',
        height=370,
        margin=dict(t=60, b=40, l=40, r=40)
    )
    fig_sexo.write_html(RUTA_GRAFICOS / 'm02_sexo.html', include_plotlyjs='cdn')
    print('✅ Gráfico sexo')

# --- Gráfico 6 (NUEVO): Barras apiladas curso × sexo ---
if 'sexo' in df.columns and curso_col:
    # Mapear sexo a etiquetas
    df_temp = df[[curso_col, 'sexo']].copy()
    df_temp['sexo_label'] = df_temp['sexo'].map({1: 'Hombre', 2: 'Mujer'})
    df_temp[curso_col] = df_temp[curso_col].astype(str)

    # Tabla cruzada curso × sexo
    cross = pd.crosstab(df_temp[curso_col], df_temp['sexo_label'])
    cross = cross.sort_index()

    fig_curso_sexo = go.Figure()
    colores_sexo = {'Hombre': '#9567FE', 'Mujer': '#FD6320'}  # Consistente con donut
    for sexo_label in ['Hombre', 'Mujer']:
        if sexo_label in cross.columns:
            fig_curso_sexo.add_trace(go.Bar(
                x=cross.index,
                y=cross[sexo_label],
                name=sexo_label,
                marker_color=colores_sexo[sexo_label]
            ))

    fig_curso_sexo.update_layout(
        barmode='stack',
        title='👥 Matrículas por Curso y Sexo (apiladas)',
        xaxis_title='Curso Académico',
        yaxis_title='Matrículas',
        height=380,
        margin=dict(t=60, b=50, l=60, r=40),
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
    )
    fig_curso_sexo.write_html(RUTA_GRAFICOS / 'm02_curso_sexo.html', include_plotlyjs='cdn')
    print('✅ Gráfico barras apiladas curso×sexo')

✅ Gráfico sexo
✅ Gráfico barras apiladas curso×sexo


In [6]:
# ============================================================================
# CELDA 7: DETECCIÓN DE ANOMALÍAS EN COLUMNAS DE TEXTO
# ============================================================================
# Analiza todas las columnas object buscando:
# - Cadenas vacías
# - Espacios al inicio/final
# - Dobles espacios
# Genera: barras con total anomalías + tabla HTML con desglose.
# ============================================================================

print('\n📊 Analizando anomalías en texto...')

text_cols = df.select_dtypes(include=['object']).columns.tolist()

anomalias_texto = []
for col in text_cols:
    serie = df[col].dropna().astype(str)
    if len(serie) == 0:
        continue

    n_vacios = (serie == '').sum()
    n_espacio_inicio = serie.str.startswith(' ').sum()
    n_espacio_fin = serie.str.endswith(' ').sum()
    n_doble_espacio = serie.str.contains('  ').sum()

    total = n_vacios + n_espacio_inicio + n_espacio_fin + n_doble_espacio
    if total > 0:
        anomalias_texto.append({
            'Columna': col,
            'Vacíos': n_vacios,
            'Espacio inicio': n_espacio_inicio,
            'Espacio fin': n_espacio_fin,
            'Doble espacio': n_doble_espacio,
            'Total': total
        })

if anomalias_texto:
    df_anom = pd.DataFrame(anomalias_texto).sort_values('Total', ascending=True)

    # --- Gráfico: Barras horizontales con total de anomalías ---
    fig_anom_barras = go.Figure(go.Bar(
        x=df_anom['Total'].values,
        y=df_anom['Columna'].values,
        orientation='h',
        marker_color=['#e53e3e' if v > 500 else '#ed8936' if v > 100 else '#38a169' for v in df_anom['Total']],
        text=[f'{v:,}' for v in df_anom['Total'].values],
        textposition='outside'
    ))
    fig_anom_barras.update_layout(
        title='🔍 Anomalías en Texto: Total por Columna<br>'
              '<span style="font-size:11px; color:#718096;">'
              '🔴 >500 | 🟠 100-500 | 🟢 <100</span>',
        xaxis_title='Nº de Anomalías',
        height=max(250, len(df_anom) * 60 + 100),
        margin=dict(t=80, b=50, l=150, r=80)
    )
    fig_anom_barras.update_xaxes(range=[0, df_anom['Total'].max() * 1.2])
    fig_anom_barras.write_html(RUTA_GRAFICOS / 'm02_anomalias_barras.html', include_plotlyjs='cdn')
    print(f'✅ Barras anomalías: {len(df_anom)} columnas')

    # --- Tabla HTML con desglose por tipo ---
    df_anom_tabla = df_anom.sort_values('Total', ascending=False)
    tabla_anom_html = '<table style="width:100%; border-collapse:collapse; font-size:0.9em;">'
    tabla_anom_html += '<thead><tr style="background:#2B6CB0; color:white;">'
    for col_name in ['Columna', 'Vacíos', 'Esp. inicio', 'Esp. fin', 'Doble esp.', 'Total']:
        tabla_anom_html += f'<th style="padding:10px 12px; text-align:center;">{col_name}</th>'
    tabla_anom_html += '</tr></thead><tbody>'

    for _, row in df_anom_tabla.iterrows():
        # Color de fondo según severidad
        if row['Total'] > 500:
            bg = '#FFF5F5'  # Rojo suave
        elif row['Total'] > 100:
            bg = '#FFFAF0'  # Naranja suave
        else:
            bg = '#F0FFF4'  # Verde suave

        tabla_anom_html += f'<tr style="background:{bg}; border-bottom:1px solid #e2e8f0;">'
        tabla_anom_html += f'<td style="padding:8px 12px; font-weight:bold;">{row["Columna"]}</td>'
        for campo in ['Vacíos', 'Espacio inicio', 'Espacio fin', 'Doble espacio']:
            val = int(row[campo])
            color = '#e53e3e' if val > 0 else '#a0aec0'
            tabla_anom_html += f'<td style="padding:8px 12px; text-align:center; color:{color}; font-weight:{"bold" if val > 0 else "normal"};">{val:,}</td>'
        tabla_anom_html += f'<td style="padding:8px 12px; text-align:center; font-weight:bold; color:#2B6CB0;">{int(row["Total"]):,}</td>'
        tabla_anom_html += '</tr>'

    tabla_anom_html += '</tbody></table>'

    # Guardar tabla como HTML independiente (para iframe)
    tabla_html_completa = f'''<!DOCTYPE html>
<html><head><meta charset="utf-8">
<style>body {{ font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif; margin:15px; }}
table {{ box-shadow: 0 1px 3px rgba(0,0,0,0.1); border-radius:8px; overflow:hidden; }}
th {{ position:sticky; top:0; }}
tr:hover {{ background:#EBF8FF !important; }}</style>
</head><body>
<h3 style="color:#2d3748; margin-bottom:12px;">📋 Desglose de Anomalías por Tipo</h3>
{tabla_anom_html}
<p style="color:#a0aec0; font-size:0.8em; margin-top:12px;">
Vacíos = cadenas "". Esp. inicio/fin = espacios extra. Doble esp. = dos espacios seguidos.
</p></body></html>'''

    ruta_tabla = RUTA_GRAFICOS / 'm02_anomalias_tabla.html'
    with open(ruta_tabla, 'w', encoding='utf-8') as f:
        f.write(tabla_html_completa)
    print(f'✅ Tabla anomalías guardada')

    for _, row in df_anom_tabla.iterrows():
        print(f'   - {row["Columna"]}: {int(row["Total"])} anomalías')
else:
    print('✅ Sin anomalías detectadas en texto')
    # Crear gráfico vacío
    fig_anom_barras = go.Figure()
    fig_anom_barras.add_annotation(text='✅ Sin anomalías en texto', x=0.5, y=0.5, showarrow=False, font_size=20)
    fig_anom_barras.update_layout(height=250, title='🔍 Anomalías en Texto')
    fig_anom_barras.write_html(RUTA_GRAFICOS / 'm02_anomalias_barras.html', include_plotlyjs='cdn')


📊 Analizando anomalías en texto...
✅ Barras anomalías: 3 columnas
✅ Tabla anomalías guardada
   - via_acceso_exp: 822 anomalías
   - titulacion: 369 anomalías
   - pais_domicilio: 7 anomalías


In [7]:
# ============================================================================
# CELDA 8: GENERAR HTML — PÁGINA M02 CALIDAD
# ============================================================================
# Ensambla la página con 5 filas:
# - KPIs + sección duplicados
# - Fila 1: Matrículas por curso + Cursos por alumno
# - Fila 2: Titulaciones + Rama
# - Fila 3: Sexo + Barras apiladas curso×sexo (NUEVO)
# - Fila 4: Anomalías texto
# ============================================================================

print('=' * 60)
print('GENERANDO HTML')
print('=' * 60)

nav_fases_html, nav_modulos_html = generar_html_navegacion_completa(
    fase_activa="fase2", modulo_activo="m02"
)

# --- KPIs ---
KPIS = [
    {'valor': formato_numero_es(n_filas), 'titulo': 'Registros'},
    {'valor': formato_numero_es(n_alumnos), 'titulo': 'Alumnos únicos'},
    {'valor': str(n_dup_exactos), 'titulo': 'Duplicados exactos'},
    {'valor': f'{filas_por_alumno:.1f}', 'titulo': 'Filas/Alumno'},
]
kpis_html = generar_kpis_html(KPIS)

# --- Sección: Duplicados ---
seccion_dup = generar_seccion_html(
    titulo="Análisis de Duplicados",
    contenido=f'''
        <div style="display:grid; grid-template-columns:1fr 1fr; gap:20px;">
            <div style="background:#f0fff4; padding:15px; border-radius:8px;">
                <strong>Duplicados exactos:</strong> {n_dup_exactos}
                <br><span style="color:#38a169;">{'✅ OK' if n_dup_exactos == 0 else '⚠️ Revisar'}</span>
            </div>
            <div style="background:#f0f9ff; padding:15px; border-radius:8px;">
                <strong>Duplicados por clave:</strong> {n_dup_clave}
                <br><span style="color:#3182ce;">{'✅ OK' if n_dup_clave == 0 else '⚠️ Revisar'}</span>
            </div>
        </div>
    ''',
    icono="🔍"
)

# --- Fila 1: Matrículas por curso + Cursos por alumno ---
seccion_fila1 = generar_seccion_html(
    titulo="Estructura Longitudinal",
    contenido='''
        <div style="display:grid; grid-template-columns:1fr 1fr; gap:20px;">
            <iframe src="graficos/m02_cursos.html" width="100%" height="370" frameborder="0"></iframe>
            <iframe src="graficos/m02_filas_alumno.html" width="100%" height="370" frameborder="0"></iframe>
        </div>
    ''',
    icono="📈"
)

# --- Fila 2: Titulaciones + Rama ---
seccion_fila2 = generar_seccion_html(
    titulo="Distribución Académica",
    contenido='''
        <div style="display:grid; grid-template-columns:1fr 1fr; gap:20px;">
            <iframe src="graficos/m02_titulaciones.html" width="100%" height="520" frameborder="0"></iframe>
            <iframe src="graficos/m02_rama.html" width="100%" height="520" frameborder="0"></iframe>
        </div>
    ''',
    icono="🎓"
)

# --- Fila 3: Sexo + Barras apiladas (NUEVO) ---
seccion_fila3 = generar_seccion_html(
    titulo="Distribución por Sexo",
    contenido='''
        <div style="display:grid; grid-template-columns:1fr 1fr; gap:20px;">
            <iframe src="graficos/m02_sexo.html" width="100%" height="390" frameborder="0"></iframe>
            <iframe src="graficos/m02_curso_sexo.html" width="100%" height="390" frameborder="0"></iframe>
        </div>
    ''',
    icono="👥"
)

# --- Fila 4: Anomalías texto (barras + tabla desglose) ---
seccion_fila4 = generar_seccion_html(
    titulo="Anomalías en Texto",
    contenido='''
        <div style="display:grid; grid-template-columns:1fr 1fr; gap:20px;">
            <iframe src="graficos/m02_anomalias_barras.html" width="100%" height="350" frameborder="0"></iframe>
            <iframe src="graficos/m02_anomalias_tabla.html" width="100%" height="350" frameborder="0"></iframe>
        </div>
    ''',
    icono="🔍"
)

# --- Ensamblar y guardar ---
contenido_html = kpis_html + seccion_dup + seccion_fila1 + seccion_fila2 + seccion_fila3 + seccion_fila4

html_completo = render_base_html(
    titulo="M02: Calidad", icono="✅",
    subtitulo="Fase 2: EDA Datos Originales | TFM Abandono Universitario",
    nav_fases=nav_fases_html, nav_modulos=nav_modulos_html, contenido=contenido_html,
    notebook_nombre='f2_m02_calidad.ipynb', notebook_carpeta='fase2_eda'
)

ruta_html = RUTA_FASE2 / "m02_calidad.html"
guardar_html(html_completo, ruta_html)
print(f"\n✅ HTML: {ruta_html}")

GENERANDO HTML
✅ HTML guardado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase2\m02_calidad.html

✅ HTML: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase2\m02_calidad.html


In [8]:
# ============================================================================
# CELDA 9: RESUMEN DE EJECUCIÓN
# ============================================================================

print('\n' + '=' * 60)
print('✅ F2-M02 COMPLETADO')
print('=' * 60)
print('📊 Fila 1: Matrículas por curso + Cursos por alumno')
print('📊 Fila 2: Titulaciones + Rama')
print('👥 Fila 3: Sexo (donut) + Barras apiladas curso×sexo')
print('🔍 Fila 4: Anomalías en texto')
print('📌 Siguiente: f2_m03_nulos.ipynb')


✅ F2-M02 COMPLETADO
📊 Fila 1: Matrículas por curso + Cursos por alumno
📊 Fila 2: Titulaciones + Rama
👥 Fila 3: Sexo (donut) + Barras apiladas curso×sexo
🔍 Fila 4: Anomalías en texto
📌 Siguiente: f2_m03_nulos.ipynb
